# Pangenome Analysis Notebook

This blueprint shows how to @TODO 

Graph genomes and Dr. Benedict Paten's lab at the University of California, Santa Cruz (UCSC)

## Download the dataset

We will use the HPRC ([Human Pangenome Reference Consortium](https://humanpangenome.org/)) version 1.1 graph. It was constructed using [Minigraph-Cactus](https://github.com/ComparativeGenomicsToolkit/cactus/blob/master/doc/pangenome.md). Specifically, we will need the following files: 
- `.gbz` - The compressed graph 
- `.dist` - The distance index used for computing distances between points in the graph 
- `.min` - The minimizer index used to find minimizer substrings in the graph 

Make data directory (if it doesn't exist)

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data
mkdir -p ${DATA_DIR}

Download the data

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data

# Donwload index files
cd ${DATA_DIR} && \
    echo -e "dist\nmin\ngbz" | xargs -I {} \
    wget "https://s3-us-west-2.amazonaws.com/human-pangenomics/pangenomes/freeze/freeze1/minigraph-cactus/hprc-v1.1-mc-grch38/hprc-v1.1-mc-grch38.{}"


## Pre-process the data

Next we will extract the reference path using [vg](https://github.com/vgteam/vg) paths and write the output to a FASTA file so we can use standard variant callers downstream. 

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data

docker run --rm --volume ${DATA_DIR}:/workdir \
    --workdir /workdir \
    quay.io/vgteam/vg:v1.59.0 \
    vg paths -x hprc-v1.1-mc-grch38.gbz -p hprc-v1.1-mc-grch38.paths.sub -F > hprc-v1.1-mc-grch38.fa


This FASTA file still needs to be indexed. 

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data

samtools faidx ${DATA_DIR}/hprc-v1.1-mc-grch38.fa

## Run Giraffe

GPU accelerated Giraffe using [Parabricks Giraffe](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_giraffe.html). 

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data

# This command assumes all the inputs are in the current working directory and all the outputs go to the same place.
docker run --rm --gpus all \
    --volume ${DATA_DIR}:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun giraffe \
    --read-group "sample_rg1" \
    --sample "sample-name" \
    --read-group-library "library" \
    --read-group-platform "platform" \
    --read-group-pu "pu" \
    --dist-name hprc-v1.1-mc-grch38.dist \
    --minimizer-name hprc-v1.1-mc-grch38.min \
    --gbz-name hprc-v1.1-mc-grch38.gbz \
    --ref-paths hprc-v1.1-mc-grch38.paths.sub \
    --in-fq ${INPUT_FASTQ_1} ${INPUT_FASTQ_2} \
    --out-bam ${OUTPUT_BAM}

## Run Pangenome-Aware DeepVariant

GPU accelerated [Parabricks Pangenome-Aware DeepVariant](https://docs.nvidia.com/clara/parabricks/latest/documentation/tooldocs/man_pangenome_aware_deepvariant.html). 

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data

docker run --rm --gpus all \
    --volume ${DATA_DIR}:/workdir \
    --workdir /workdir \
    nvcr.io/nvidia/clara/clara-parabricks:4.6.0-1 \
    pbrun pangenome_aware_deepvariant \
    --ref /workdir/hprc-v1.1-mc-grch38.fa \
    --pangenome /workdir/hprc-v1.1-mc-grch38.gbz \
    --in-bam /workdir/${INPUT_BAM} \
    --out-variants /outputdir/${OUTPUT_VCF}

Let's check the VCF output using bcftools

In [ ]:
%%sh 

DATA_DIR=$(pwd)/data

docker run --rm \
    --volume ${DATA_DIR}:/workdir \
    biocontainers/bcftools:v1.5_cv3 \
    bcftools view --header data/out/deepvariant.annotated.vcf.gz | tail